# MotoGP Constructors - Advanced Modeling and Analysis

**Dataset Focus**: `constructure_world_championship.csv`  
**CRISP-DM Phase**: 4 - Modeling  
**Purpose**: Advanced statistical modeling of constructor championships and dominance patterns

## Business Focus
- Constructor dominance analysis across eras
- Championship efficiency modeling
- Multi-class success patterns

In [ ]:
# Setup and data loading
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Load prepared constructors data
data_path = Path("../../data/interim/")
df = pd.read_csv(data_path / "constructors_prepared.csv")

print(f"Constructors data loaded: {df.shape}")
print(f"Date range: {df['season'].min()} - {df['season'].max()}")
print(f"Unique constructors: {df['constructor_clean'].nunique()}")
print(f"Unique classes: {df['class_clean'].nunique()}")
df.head()

## Constructor Dominance Modeling

In [ ]:
print("=== CONSTRUCTOR DOMINANCE ANALYSIS ===")

# Calculate dominance metrics
constructor_metrics = df.groupby('constructor_clean').agg({
    'season': ['count', 'min', 'max', 'nunique'],
    'class_clean': 'nunique',
    'class_category': 'nunique'
})

constructor_metrics.columns = ['total_championships', 'first_championship', 'last_championship', 
                              'seasons_active', 'classes_won', 'categories_won']

constructor_metrics['championship_span'] = constructor_metrics['last_championship'] - constructor_metrics['first_championship']
constructor_metrics['championships_per_season'] = constructor_metrics['total_championships'] / constructor_metrics['seasons_active']
constructor_metrics['versatility_score'] = constructor_metrics['categories_won'] * constructor_metrics['classes_won']

# Sort by total championships
constructor_metrics = constructor_metrics.sort_values('total_championships', ascending=False)

print("Top 10 constructor performance metrics:")
print(constructor_metrics.head(10).round(2))

# Dominance categories
def categorize_constructor(row):
    if row['total_championships'] >= 20:
        return 'Legendary Dominance'
    elif row['total_championships'] >= 10:
        return 'Major Success'
    elif row['total_championships'] >= 5:
        return 'Consistent Winner'
    elif row['categories_won'] >= 2:
        return 'Versatile Competitor'
    else:
        return 'Specialized Success'

constructor_metrics['dominance_category'] = constructor_metrics.apply(categorize_constructor, axis=1)

print(f"\nDominance categories:")
print(constructor_metrics['dominance_category'].value_counts())

# Visualization
plt.figure(figsize=(14, 8))
top_15 = constructor_metrics.head(15)
plt.barh(range(len(top_15)), top_15['total_championships'], color='gold')
plt.yticks(range(len(top_15)), top_15.index)
plt.xlabel('Total Championships')
plt.title('Top 15 Constructor Championship Totals')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Era-based Analysis

In [ ]:
print("=== ERA-BASED CONSTRUCTOR ANALYSIS ===")

# Analyze constructor performance by era
era_analysis = df.groupby(['era', 'constructor_clean']).size().reset_index(name='championships')
era_dominance = era_analysis.groupby('era').apply(
    lambda x: x.nlargest(3, 'championships')[['constructor_clean', 'championships']]
).reset_index(drop=True)

print("Top 3 constructors by era:")
for era in df['era'].unique():
    era_data = era_analysis[era_analysis['era'] == era]
    top_3 = era_data.nlargest(3, 'championships')
    print(f"\n{era}:")
    for i, (_, constructor) in enumerate(top_3.iterrows(), 1):
        print(f"  {i}. {constructor['constructor_clean']}: {constructor['championships']} championships")

# Championship concentration analysis
print(f"\n=== CHAMPIONSHIP CONCENTRATION ANALYSIS ===")
for era in df['era'].unique():
    era_data = era_analysis[era_analysis['era'] == era]
    total_championships = era_data['championships'].sum()
    top_constructor_share = era_data['championships'].max() / total_championships * 100
    hhi = sum((champ/total_championships)**2 for champ in era_data['championships'])
    
    print(f"{era}: {top_constructor_share:.1f}% concentration, HHI: {hhi:.3f}")

# Visualization: Era transition
pivot_era = df.groupby(['decade', 'constructor_clean']).size().unstack(fill_value=0)
top_constructors = constructor_metrics.head(5).index
plt.figure(figsize=(14, 8))
for constructor in top_constructors:
    if constructor in pivot_era.columns:
        plt.plot(pivot_era.index, pivot_era[constructor], marker='o', label=constructor)
plt.xlabel('Decade')
plt.ylabel('Championships')
plt.title('Constructor Championship Evolution by Decade')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Class Success Modeling

In [ ]:
print("=== CLASS SUCCESS MODELING ===")

# Multi-class success analysis
class_success = df.groupby(['constructor_clean', 'class_category']).size().unstack(fill_value=0)
class_success['total'] = class_success.sum(axis=1)
class_success['versatility'] = (class_success > 0).sum(axis=1)

# Most versatile constructors
most_versatile = class_success.nlargest(10, 'versatility')
print("Most versatile constructors (across class categories):")
print(most_versatile[['versatility', 'total']].head(10))

# Class dominance
class_champions = {}
for class_cat in df['class_category'].unique():
    class_data = df[df['class_category'] == class_cat]
    champion = class_data['constructor_clean'].value_counts().index[0]
    championships = class_data['constructor_clean'].value_counts().iloc[0]
    class_champions[class_cat] = (champion, championships)

print(f"\nClass category champions:")
for class_cat, (champion, count) in class_champions.items():
    print(f"{class_cat}: {champion} ({count} championships)")

# Clustering analysis
if len(class_success) >= 10:
    print(f"\n=== CONSTRUCTOR CLUSTERING ===")
    
    # Prepare features for clustering
    cluster_features = ['total', 'versatility']
    for col in class_success.columns:
        if col not in ['total', 'versatility']:
            cluster_features.append(col)
    
    X = class_success[cluster_features].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # K-means clustering
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    class_success['cluster'] = clusters
    
    print("Constructor clusters:")
    for cluster_id in range(4):
        cluster_members = class_success[class_success['cluster'] == cluster_id]
        print(f"\nCluster {cluster_id}: {len(cluster_members)} constructors")
        print(f"  Average championships: {cluster_members['total'].mean():.1f}")
        print(f"  Average versatility: {cluster_members['versatility'].mean():.1f}")
        top_members = cluster_members.nlargest(3, 'total')
        print(f"  Top members: {list(top_members.index)[:3]}")

print(f"\n✅ CONSTRUCTORS MODELING COMPLETED")
print(f"Key insights: Constructor dominance varies by era, with clear hierarchies")
print(f"and specialization patterns across different racing categories.")